# How the posterior moves with the error

One filter step, for one weight. Gaussian prior, Laplacian noise ($\beta^* = 1$).

**Setup**
- Three weights, $w_{t-1} = [0,0,0]$. So $e = d - w_{t-1}^\top x_t = d$, and the prior is centred at 0.
- $x_t = [1,1,1]$. The three weights behave the same, so we look at one of them: $x_m = 1$.
- $\tilde v = 0.0438$, $b_\eta = 0.158$ (same values as `gain-functions.ipynb`).

**Posterior** (eq. 27 with $w_{t-1} = 0$):
$$f(\theta \mid e) \;\propto\; \underbrace{\mathcal N(\theta;\,0,\,\tilde v)}_{\text{prior}} \;\cdot\; \underbrace{f^0_{\zeta_m}(e - x_m\theta)}_{\text{likelihood, eq. 56}}$$

The likelihood is the Laplacian noise, blurred by the uncertainty of the other two weights (eq. 56):
$$f^0_{\zeta_m}(u) = \frac{e^{\sigma_m^2/(2b_\eta^2)}}{2b_\eta}\left[e^{-u/b_\eta}\,\Phi\!\left(\frac{u}{\sigma_m}-\frac{\sigma_m}{b_\eta}\right) + e^{u/b_\eta}\,\Phi\!\left(-\frac{u}{\sigma_m}-\frac{\sigma_m}{b_\eta}\right)\right],
\qquad \sigma_m^2 = \tilde v\,\|x_{t,\setminus m}\|^2$$

**Question:** as $e$ grows, where do the mean and the mode of the posterior go?

With these values σ_m is 1.9·b_η, which blurs the Laplacian a lot, so mean and mode look almost the same. If we want to see gap between them, a smaller σ_m would show it, but that needs a different x_t.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.special import log_ndtr
from ipywidgets import interact, FloatSlider

plt.rcParams.update({"font.size": 13})
COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]

v_tilde = 0.043831                  # predicted variance of each weight
b_eta = 0.158114                    # Laplacian noise scale
x_t = np.array([1.0, 0.1, 0.1])     # SIGMA_m !!!
m = 0                               # the weight we look at
x_m = x_t[m]

sigma_m = np.sqrt(v_tilde * (x_t @ x_t - x_m**2))   # blur added by the other two weights
a_m = v_tilde * x_m / b_eta                        # where the mean saturates (eq. 57)

print(f"sigma_m = {sigma_m:.3f}")
print(f"a_m     = {a_m:.3f}")

## 1. Prior, likelihood, posterior

Multiplying very small numbers can round to zero, so we work with logs:
1. log prior + log likelihood (both written directly as logs),
2. `exp` to go back,
3. divide by the area, so the posterior integrates to 1.

Same idea as `sKF_L_integral_algorithm` in `filters.py`.

In [ ]:
E_MAX = 20 * b_eta                                   # largest error we look at
theta = np.arange(-2.0, E_MAX / x_m + 2.0, 5e-4)     # grid for the weight


def log_prior(th):
    return -th**2 / (2 * v_tilde) - 0.5 * np.log(2 * np.pi * v_tilde)


def log_likelihood(u):                               # log of eq. 56
    s, b = sigma_m, b_eta
    return (s**2 / (2 * b**2) - np.log(2 * b)
            + np.logaddexp(-u / b + log_ndtr(u / s - s / b),
                           u / b + log_ndtr(-u / s - s / b)))


def to_density(log_p):                               # back from logs, area = 1
    p = np.exp(log_p - log_p.max())                  # subtract the max first, so the peak is 1
    return p / np.trapezoid(p, theta)


def posterior(e):
    return to_density(log_prior(theta) + log_likelihood(e - x_m * theta))


def mean_and_mode(p):
    return np.trapezoid(theta * p, theta), theta[np.argmax(p)]

## 2. Saturation error

For large $e$ the mean stops growing and approaches $a_m = \tilde v\,x_m/b_\eta$.
The **saturation error** is the first $e$ where the mean reaches 99 % of $a_m$. The slider goes a bit past it.

In [ ]:
e_grid = np.linspace(0, E_MAX, 801)
mean_e, mode_e = np.array([mean_and_mode(posterior(e)) for e in e_grid]).T

e_sat = e_grid[np.argmax(mean_e >= 0.99 * a_m)]
e_slider_max = 1.25 * e_sat
print(f"saturation error: e = {e_sat:.2f}  (= {e_sat / b_eta:.1f} b_eta)")

## 3. Move the slider

- **Left:** prior (fixed), likelihood (its peak is at $\theta = e/x_m$), posterior, with its mean and mode.
- **Right:** mean and mode for every $e$. The dots mark the current $e$.

In [ ]:
Y_MAX = 1.1 * posterior(0).max()


def show(e):
    p = posterior(e)
    mean, mode = mean_and_mode(p)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)

    ax1.semilogy(theta, np.exp(log_prior(theta)), color=COLORS[0], lw=2, label="prior")
    ax1.semilogy(
        theta,
        np.exp(log_likelihood(e - x_m * theta)),
        color=COLORS[1],
        lw=2,
        label="likelihood (eq. 56)",
    )
    ax1.plot(theta, p, color="k", lw=2.5, label="posterior")
    ax1.axvline(mean, color=COLORS[2], ls="--", lw=2, label=f"mean = {mean:.3f}")
    ax1.axvline(mode, color=COLORS[3], ls=":", lw=2, label=f"mode = {mode:.3f}")
    ax1.axvline(a_m, color="gray", lw=1, label=f"$a_m$ = {a_m:.3f}")
    ax1.set(xlim=(-1, e_slider_max / x_m + 1), ylim=(1e-10, Y_MAX), xlabel=r"$\theta$", title=f"e = {e:.3f}")
    ax1.legend(loc="upper right")

    ax2.plot(e_grid, mean_e, color=COLORS[2], ls="--", lw=2, label="mean")
    ax2.plot(e_grid, mode_e, color=COLORS[3], ls=":", lw=2, label="mode")
    ax2.plot([e], [mean], "o", color=COLORS[2], ms=14, mfc="none", mew=2.5)
    ax2.plot([e], [mode], "o", color=COLORS[3], ms=7)
    ax2.axhline(a_m, color="gray", lw=1, label="$a_m$")
    ax2.axvline(e_sat, color="gray", ls="--", lw=1, label="saturation error")
    ax2.set(xlim=(0, e_slider_max), ylim=(0, 1.15 * a_m), xlabel="$e$", ylabel=r"$\theta$",
            title="mean and mode vs e")
    ax2.grid(alpha=0.3)
    ax2.legend(loc="lower right")
    plt.show()


interact(show, e=FloatSlider(value=0.3, min=0, max=e_slider_max, step=e_slider_max / 200,
                             readout_format=".3f", layout={"width": "90%"}));

**What we see**
- **Small $e$:** mean and mode grow almost in proportion to $e$. The posterior follows the data, part of the way.
- **Large $e$:** both stop at $a_m = 0.277$ (from $e \approx 1.64$, about $10\,b_\eta$). The likelihood keeps moving right, the posterior does not follow. One observation can move the weight by at most $a_m$: this is the robustness to outliers.
- **Mean vs mode:** the mode is always slightly ahead of the mean, by at most 0.007 (near $e \approx 1.1$). With these values they almost coincide.
- **Width:** at $e = 0$ the posterior is narrower than the prior (std 0.18 vs 0.21). At large $e$ it goes back to the prior width: a very large error says nothing new about the weight.

## 4. Is eq. 57 right?

Eq. 57 writes the same posterior as a sum of two terms, one per side of the Laplacian ($\varsigma = \pm 1$):
$$f(\theta \mid e) \;\propto\; \sum_{\varsigma=\pm1} e^{-\varsigma e/b_\eta}\;\mathcal N(\theta;\,\varsigma a_m,\,\tilde v)\;\Phi(\alpha_\varsigma + \beta_\varsigma\theta),
\qquad \alpha_\varsigma = \frac{\varsigma e}{\sigma_m} - \frac{\sigma_m}{b_\eta},\quad \beta_\varsigma = -\frac{\varsigma x_m}{\sigma_m}$$

(The draft writes $d = \theta - w_{t-1,m}$; here $w_{t-1} = 0$, so $d = \theta$.)

We build it the same way (logs, `exp`, divide by the area) and compare it with the posterior of section 1, for every $e$.

In [ ]:
def posterior_57(e):
    terms = []
    for s in (+1, -1):
        alpha = s * e / sigma_m - sigma_m / b_eta
        beta = -s * x_m / sigma_m
        log_normal = -(theta - s * a_m)**2 / (2 * v_tilde) - 0.5 * np.log(2 * np.pi * v_tilde)
        terms.append(-s * e / b_eta + log_normal + log_ndtr(alpha + beta * theta))
    return to_density(np.logaddexp(*terms))


diff = np.array([np.max(np.abs(posterior(e) - posterior_57(e))) / np.max(posterior(e)) for e in e_grid])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for i, e in enumerate([0.0, e_sat / 2, e_sat]):
    ax1.plot(theta, posterior(e), color=COLORS[i], lw=5, alpha=0.35, label=f"eq. 56, e = {e:.2f}")
    ax1.plot(theta, posterior_57(e), color=COLORS[i], ls="--", lw=1.5, label=f"eq. 57, e = {e:.2f}")
ax1.set(xlim=(-1, 1.2), xlabel=r"$\theta$", title="posterior: eq. 56 (thick) and eq. 57 (dashed)")
ax1.legend(fontsize=10)

ax2.semilogy(e_grid, diff, color="k", lw=2)
ax2.set(xlim=(0, E_MAX), ylim=(1e-18, 1), xlabel="$e$", ylabel="max difference / posterior peak",
        title="difference between the two")
ax2.grid(alpha=0.3)
plt.show()

print(f"largest difference over all e: {diff.max():.1e} of the posterior peak")
print("-> eq. 57 matches eq. 56" if diff.max() < 1e-10 else "-> eq. 57 does NOT match eq. 56")